# Understanding the Roman WFI Reference Pixel Reference File

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to understand the content and purpose of the **Reference Pixel** (`REFPIX`) reference file.

The REFPIX file contains frequency-dependent coefficients used by the `romancal.refpix.RefPixStep` to correct for correlated 1/f noise (horizontal banding) using the Improved Roman Reference Correction (IRRC) algorithm. It leverages the reference pixels (border reference rows/columns and amplifier 33) present in every WFI exposure.

More details about this and other reference files can be found in the [Reference File Information](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/index.html).

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide important information about setting up your environment and installing dependencies.

## Imports
Libraries used:
- *astropy* for image normalization
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions

In [ ]:
import os
from astropy.visualization import simple_norm
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm

### The Calibration Reference Data System (CRDS)

The reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. For more information about how CRDS works and how it assigns the most appropriate reference file for each calibration step, refer to the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb). 

**IMPORTANT NOTE:** Reference files are a work in progress and will be updated several times before Roman launch. If you notice irregularities or missing information, please understand that they may be a known issue. If you have questions, please contact the [Roman Help Desk](https://romanhelp.stsci.edu).

In [ ]:
import crds

Now let's dive into this reference file type.

### Reference Pixels

The REFPIX reference file is used in the `romancal.refpix.RefPixStep()` to remove correlated 1/f noise from the raw data. The correction uses the reference pixels (4-pixel-wide border rows/columns on all four sides and the virtual 33rd amplifier) together with frequency-dependent coefficients stored in this reference file.

The current reference files were derived from TVAC1 data and significantly reduce the 1/f noise (typically by a factor of ~20).

For more details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/latest/roman/refpix/index.html) and the technical report on the IRRC algorithm and
[Rdox documentation](https://roman-docs.stsci.edu/data-handbook/roman-wfi-data-pipelines/exposure-level-pipeline#ExposureLevelPipeline-refpix) for the reference pixel correction.


Before proceeding, let's check the environmental variables set for CRDS

In [ ]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

If we want to change the context, we can do it in the next cell. In this case, we choose context `roman_0055.pmap`.

In [ ]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

### Retrieving Reference Files

As you run the exposure pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, these can be retrieved through the `crds` Python API.

For the REFPIX files, the required keywords are typically:
- `ROMAN.META.INSTRUMENT.NAME`
- `ROMAN.META.INSTRUMENT.DETECTOR`
- `ROMAN.META.EXPOSURE.START_TIME`

These keywords may be combined into a single dictionary that is later fed to the  `crds.getrecommendations` function. This function returns a dictionary of file names that match the criteria that you supply. 

In [ ]:
meta = {
    'ROMAN.META.INSTRUMENT.NAME': 'WFI',
    'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
    'ROMAN.META.EXPOSURE.TYPE': 'WFI_IMAGE',
    'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
}

ref_files = crds.getreferences(meta, reftypes=['refpix'], observatory='roman')

And once again we can examine the output of `ref_files`:

In [ ]:
ref_files

We can also use `crds.getreferences()` to accomplish the same thing; however, `getreferences()` goes one step further beyond `getrecommendations()` and will download the reference files if they are not already in your local cache. Using the same example as above:

In [ ]:
ref_files = crds.getreferences(meta, reftypes=['refpix'], observatory='roman')

And once again we can examine the output of `ref_files`:

In [ ]:
ref_files

### Examining Reference Files

Reference files use `roman_datamodels` just like WFI science data products. Let's take a closer look at the REFPIX file:


In [ ]:
refpix = rdm.open(ref_files['refpix'])
refpix.info()

The REFPIX reference file typically contains metadata and arrays of frequency-dependent coefficients (weights) used by the IRRC algorithm for the different reference signals:
* `alpha`: Reference output (amplifier 33) correction coefficients
* `gamma`: Left column correction coefficients
* `zeta`: Right column correction coefficients

Let's check the shape of these arrays:

In [ ]:
print("refpix shape information:")
for attr in ['data', 'alpha', 'gamma', 'zeta', 'coefficients']:
    if hasattr(refpix, attr):
        arr = getattr(refpix, attr)
        if arr is not None:
            print(f"  {attr}: {arr.shape if hasattr(arr, 'shape') else type(arr)}")

Now let's get some basic statistics on the coefficient arrays (example using a representative array):

In [ ]:
# Example: inspect one of the coefficient arrays 
print("Coefficient statistics:")
for name in ['alpha', 'gamma', 'zeta']:
    if hasattr(refpix, name) and getattr(refpix, name) is not None:
        arr = getattr(refpix, name)
        print(f"  {name}: shape={arr.shape}, min={arr.min():.4f}, max={arr.max():.4f}, "
              f"mean={arr.mean():.4f}, std={arr.std():.4f}")

Let's get a quick histogram of the coefficient values:

In [ ]:

if hasattr(refpix, 'alpha') and refpix.alpha is not None:
    plt.figure(figsize=(8, 5))
    plt.hist(refpix.alpha.flatten(), bins=100, log=True)
    plt.xlabel('Coefficient Value')
    plt.ylabel('Count (log)')
    plt.title('Distribution of REFPIX Coefficients (alpha example)')
    plt.show()

Now let's plot a fast visualization example of the REFPIX coefficients

In [ ]:
fig = plt.figure(figsize=(12, 8))

# Representative subset (first 5000 bins or all if small)
max_points = 5000

for i, name in enumerate(['alpha', 'gamma', 'zeta']):
    if hasattr(refpix, name) and getattr(refpix, name) is not None:
        arr = getattr(refpix, name).flatten() 
        x = np.arange(len(arr))
        
        plt.subplot(3, 1, i+1)
        # Downsample for speed if very long
        if len(arr) > max_points:
            step = len(arr) // max_points
            plt.plot(x[::step], arr[::step], label=name.capitalize(), alpha=0.8)
            plt.title(f'{name.capitalize()} Coefficients (downsampled)')
        else:
            plt.plot(x, arr, label=name.capitalize())
            plt.title(f'{name.capitalize()} Coefficients')
        
        plt.xlabel('Frequency Bin')
        plt.ylabel('Coefficient Value')
        plt.grid(True, alpha=0.3)
        plt.legend()

plt.tight_layout()
plt.show()

We can also plot the full set of coefficient arrays (note generating this plot will take some time)

In [ ]:
plt.figure(figsize=(10, 6))
if hasattr(refpix, 'alpha') and refpix.alpha is not None:
    plt.plot(refpix.alpha, label='Alpha (Amp 33)', alpha=0.8)
if hasattr(refpix, 'gamma') and refpix.gamma is not None:
    plt.plot(refpix.gamma, label='Gamma (Left)', alpha=0.8)
if hasattr(refpix, 'zeta') and refpix.zeta is not None:
    plt.plot(refpix.zeta, label='Zeta (Right)', alpha=0.8)

plt.xlabel('Frequency Bin Index')
plt.ylabel('Correction Coefficient')
plt.title('REFPIX Frequency-Dependent Coefficients')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## About this Notebook
**Author:** T. Desjardins & R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>